In [1]:

from pathlib import Path
import pandas as pd

# Original source file — read only
DATA_PATH = Path(
    r"C:\Users\PRATIK P UPADHYE\Downloads\Retail_Ledger\Superstore.csv"
)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, encoding="cp1252")

print("Dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"File size: {DATA_PATH.stat().st_size:,} bytes")


Dataset loaded successfully.
Rows: 9,994
Columns: 21
File size: 2,298,594 bytes


In [2]:

print("Column names:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

print("\nDataset dimensions:")
print(df.shape)

print("\nData types:")
display(df.dtypes.to_frame(name="data_type"))


Column names:
['Row', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

First five rows:


,Row,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11-08-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11-08-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,06-12-2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10-11-2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10-11-2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164



Dataset dimensions:
(9994, 21)

Data types:


,data_type
Row,int64
Order ID,str
Order Date,str
Ship Date,str
Ship Mode,str
Customer ID,str
Customer Name,str
Segment,str
Country,str
City,str


In [3]:

# Count missing values in each column
missing_values = df.isna().sum().sort_values(ascending=False)

print("Missing values by column:")
display(missing_values.to_frame(name="missing_count"))

print("Total missing cells:", int(df.isna().sum().sum()))

# Check exact duplicate rows
duplicate_rows = int(df.duplicated().sum())

print(f"\nExact duplicate rows: {duplicate_rows:,}")

# Check duplicate values in the Row identifier column
duplicate_row_ids = int(df["Row"].duplicated().sum())

print(f"Duplicate Row identifiers: {duplicate_row_ids:,}")


Missing values by column:


,missing_count
Row,0
Order ID,0
Order Date,0
Ship Date,0
Ship Mode,0
Customer ID,0
Customer Name,0
Segment,0
Country,0
City,0


Total missing cells: 0

Exact duplicate rows: 0
Duplicate Row identifiers: 0


In [4]:

# Inspect numeric columns
numeric_columns = ["Sales", "Quantity", "Discount", "Profit"]

print("Numeric column data types:")
display(df[numeric_columns].dtypes.to_frame(name="data_type"))

print("\nNumeric summary statistics:")
display(df[numeric_columns].describe().T)

# Check basic business-rule violations
numeric_checks = {
    "Sales <= 0": int((df["Sales"] <= 0).sum()),
    "Quantity <= 0": int((df["Quantity"] <= 0).sum()),
    "Discount < 0": int((df["Discount"] < 0).sum()),
    "Discount > 1": int((df["Discount"] > 1).sum()),
    "Missing Profit": int(df["Profit"].isna().sum()),
}

print("\nBusiness-rule validation:")
display(
    pd.DataFrame.from_dict(
        numeric_checks, orient="index", columns=["invalid_count"]
    )
)


Numeric column data types:


,data_type
Sales,float64
Quantity,int64
Discount,float64
Profit,float64



Numeric summary statistics:


,count,mean,std,min,25%,50%,75%,max
Sales,9994.0,229.858001,623.245101,0.444,17.28000,54.4900,209.940,22638.480
Quantity,9994.0,3.789574,2.225110,1.000,2.00000,3.0000,5.000,14.000
Discount,9994.0,0.156203,0.206452,0.000,0.00000,0.2000,0.200,0.800
Profit,9994.0,28.656896,234.260108,-6599.978,1.72875,8.6665,29.364,8399.976



Business-rule validation:


,invalid_count
Sales <= 0,0
Quantity <= 0,0
Discount < 0,0
Discount > 1,0
Missing Profit,0


In [5]:

# Parse dates for validation without modifying the original columns
order_dates = pd.to_datetime(
    df["Order Date"],
    format="mixed",
    dayfirst=False,
    errors="coerce"
)

ship_dates = pd.to_datetime(
    df["Ship Date"],
    format="mixed",
    dayfirst=False,
    errors="coerce"
)

# Summarize date validation
date_checks = pd.DataFrame({
    "check": [
        "Unparsed order dates",
        "Unparsed ship dates",
        "Ship dates before order dates",
    ],
    "count": [
        int(order_dates.isna().sum()),
        int(ship_dates.isna().sum()),
        int((ship_dates < order_dates).sum()),
    ],
})

display(date_checks)

print("\nOrder date range:")
print(f"Earliest: {order_dates.min()}")
print(f"Latest:   {order_dates.max()}")

print("\nShip date range:")
print(f"Earliest: {ship_dates.min()}")
print(f"Latest:   {ship_dates.max()}")


,check,count
0,Unparsed order dates,0
1,Unparsed ship dates,0
2,Ship dates before order dates,0



Order date range:
Earliest: 2014-01-03 00:00:00
Latest:   2017-12-30 00:00:00

Ship date range:
Earliest: 2014-01-07 00:00:00
Latest:   2018-01-05 00:00:00


In [6]:

# Inspect distinct values in important categorical columns
categorical_columns = [
    "Ship Mode",
    "Segment",
    "Region",
    "Category",
    "Sub-Category",
]

for column in categorical_columns:
    print(f"\n{column} ({df[column].nunique()} distinct values):")
    print(sorted(df[column].dropna().unique()))

# Inspect postal-code formatting
postal_code_text = df["Postal Code"].astype("string").str.strip()

postal_checks = {
    "Missing postal codes": int(postal_code_text.isna().sum()),
    "Distinct postal codes": int(postal_code_text.nunique()),
    "Postal codes shorter than 5 digits": int(
        postal_code_text.str.len().lt(5).sum()
    ),
    "Postal codes equal to zero": int(
        pd.to_numeric(postal_code_text, errors="coerce").eq(0).sum()
    ),
}

print("\nPostal-code checks:")
display(
    pd.DataFrame.from_dict(
        postal_checks, orient="index", columns=["count"]
    )
)



Ship Mode (4 distinct values):
['First Class', 'Same Day', 'Second Class', 'Standard Class']

Segment (3 distinct values):
['Consumer', 'Corporate', 'Home Office']

Region (4 distinct values):
['Central', 'East', 'South', 'West']

Category (3 distinct values):
['Furniture', 'Office Supplies', 'Technology']

Sub-Category (17 distinct values):
['Accessories', 'Appliances', 'Art', 'Binders', 'Bookcases', 'Chairs', 'Copiers', 'Envelopes', 'Fasteners', 'Furnishings', 'Labels', 'Machines', 'Paper', 'Phones', 'Storage', 'Supplies', 'Tables']

Postal-code checks:


,count
Missing postal codes,0
Distinct postal codes,631
Postal codes shorter than 5 digits,449
Postal codes equal to zero,0


In [7]:

# Consolidated data-validation summary
validation_summary = pd.DataFrame([
    ("Dataset rows", len(df), "Passed"),
    ("Dataset columns", df.shape[1], "Passed"),
    ("Missing cells", int(df.isna().sum().sum()), "Passed"),
    ("Exact duplicate rows", int(df.duplicated().sum()), "Passed"),
    ("Duplicate Row identifiers", int(df["Row"].duplicated().sum()), "Passed"),
    ("Nonpositive sales", int((df["Sales"] <= 0).sum()), "Passed"),
    ("Nonpositive quantity", int((df["Quantity"] <= 0).sum()), "Passed"),
    ("Discounts outside 0–1", int(((df["Discount"] < 0) | (df["Discount"] > 1)).sum()), "Passed"),
    ("Unparsed order dates", int(order_dates.isna().sum()), "Passed"),
    ("Unparsed ship dates", int(ship_dates.isna().sum()), "Passed"),
    ("Ship dates before order dates", int((ship_dates < order_dates).sum()), "Passed"),
    ("Postal codes shorter than 5 digits", int(postal_code_text.str.len().lt(5).sum()), "Review"),
], columns=["validation_check", "result", "status"])

display(validation_summary)

print("\nValidation complete.")
print("The original CSV has not been modified.")


,validation_check,result,status
0,Dataset rows,9994,Passed
1,Dataset columns,21,Passed
2,Missing cells,0,Passed
3,Exact duplicate rows,0,Passed
4,Duplicate Row identifiers,0,Passed
5,Nonpositive sales,0,Passed
6,Nonpositive quantity,0,Passed
7,Discounts outside 0–1,0,Passed
8,Unparsed order dates,0,Passed
9,Unparsed ship dates,0,Passed



Validation complete.
The original CSV has not been modified.
